# 07 — Model 2 Tuned: Frozen ResNet + Finer-Grained Early Stopping

Model 2's real run (`06_model2_hybrid_training.ipynb`) peaked at epoch 1 (ROC-AUC 0.660, PR-AUC 0.212 -- tied with `text_only_bert` on PR-AUC!) then drifted down through epoch 4 before early stopping (patience=3 epochs) caught it. Param breakdown showed ResNet's `layer4` (14.96M trainable) is over 2x BERT's one unfrozen layer (7.09M) -- likely the bigger overfitting contributor.

Three changes via `configs/exp07_model2_hybrid_tuned.yaml`:
1. **ResNet fully frozen** (`unfreeze_image_blocks: 0`, was 1) -- only BERT's top layer + the fusion head adapt now (7.8M/150.2M trainable, 5.2% -- down from 22.8M/15.2%).
2. **Validation every 500 steps** (`eval_every_n_steps`, new `train.py` feature) instead of only at epoch end (~2730 steps) -- catches a peak this narrow more precisely.
3. **weight_decay 0.05** (was 0.01) -- more regularization given the fast-onset overfitting.

Smoke-tested before this run: param counts (0 ResNet trainable, confirmed), and the new mid-epoch eval logic (validated at steps 3 and 6 within a 1-epoch/8-step test, no duplicate end-of-epoch eval).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run training

**Do not run this at the same time as any other GPU/MPS process.**

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp07_model2_hybrid_tuned.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/hybrid_model2_tuned"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation checks logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Model 2 (tuned) training loss")

axes[1].plot(val_log["step"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["step"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.212, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.212)")
axes[1].axhline(0.212, color="green", linestyle=":", label="Model 2 (v1) best PR-AUC (0.212)")
axes[1].set_xlabel("step")
axes[1].set_title("Model 2 (tuned) validation metrics (by step, not just epoch)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

In [ ]:
best_tuned = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208},
    {"model": "text_only_bert",       "roc_auc": 0.704, "pr_auc": 0.212},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167},
    {"model": "siglip2_stage_c_lora", "roc_auc": 0.659, "pr_auc": 0.184},
    {"model": "model1_crossattn_full","roc_auc": 0.625, "pr_auc": 0.161},
    {"model": "model1b_crossattn_lora_only", "roc_auc": 0.643, "pr_auc": 0.179},
    {"model": "model2_hybrid_v1",     "roc_auc": 0.660, "pr_auc": 0.212},
    {"model": "model2_hybrid_tuned",  "roc_auc": best_tuned["roc_auc"], "pr_auc": best_tuned["pr_auc"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
